# 충남대학교 학내 정보 Q&A 시스템

**자연어처리 텀프로젝트 — 202202497 장윤상**

| 구성 요소 | 선택 |
|-----------|------|
| Base 모델 | `google/gemma-4-12b-it` (4bit NF4) |
| 임베딩 | `BAAI/bge-m3` (CPU) |
| 벡터 DB | ChromaDB |

> **사용법**: GPU(T4) 런타임 선택 → **모두 실행** → 설치 후 런타임 자동 재시작 → 다시 **모두 실행**

## 0. 소스코드 클론 & 의존성 설치

In [1]:
import os

REPO_URL = "https://github.com/adoveflash/cnu_qa_system.git"
REPO_DIR = "cnu_qa_system"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    print(f"이미 존재: {REPO_DIR}")

if not os.getcwd().endswith(REPO_DIR):
    os.chdir(REPO_DIR)
print(f"작업 디렉터리: {os.getcwd()}")

from google.colab import drive
drive.mount("/content/drive")
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

!pip install -q --upgrade transformers "bitsandbytes>=0.46.1"
!pip install -q sentence-transformers chromadb gradio accelerate \
    requests beautifulsoup4 pdfplumber huggingface_hub

try:
    import transformers
    if not hasattr(transformers, 'Gemma4ForCausalLM'):
        print("런타임 재시작 필요 — 자동 재시작합니다...")
        os.kill(os.getpid(), 9)
    else:
        print(f"transformers {transformers.__version__} OK")
except:
    pass

Cloning into 'cnu_qa_system'...
remote: Enumerating objects: 775, done.
remote: Counting objects: 100% (106/106), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 775 (delta 44), reused 57 (delta 22), pack-reused 669 (from 1)
Receiving objects: 100% (775/775), 3.52 MiB | 8.88 MiB/s, done.
Resolving deltas: 100% (438/438), done.
작업 디렉터리: /content/cnu_qa_system
Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 1. 환경 설정

In [3]:
import os, json, time, torch, random, gc, re
from datetime import datetime, timezone, timedelta
from urllib.parse import urlparse

REPO_DIR = "cnu_qa_system"
if os.path.exists(REPO_DIR) and not os.getcwd().endswith(REPO_DIR):
    os.chdir(REPO_DIR)

# Drive HF 캐시 복원 (런타임 재시작 후)
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"

SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)

print(f"작업 디렉터리: {os.getcwd()}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

작업 디렉터리: /content/cnu_qa_system
CUDA: True
GPU: Tesla T4
VRAM: 14.6 GB


## 2. 데이터 다운로드

In [4]:
from huggingface_hub import snapshot_download

HF_REPO = "adoveflash/cnu-qa-system"
VECTOR_DB_PATH = "data/vector_db"

if not os.path.exists(VECTOR_DB_PATH):
    snapshot_download(repo_id=HF_REPO, local_dir=".", allow_patterns=["data/vector_db/**"])
    print("벡터 DB 다운로드 완료")
else:
    print("벡터 DB 이미 존재")

if not os.path.exists("data/corpus/chunks.jsonl"):
    snapshot_download(repo_id=HF_REPO, local_dir=".", allow_patterns=["data/corpus/chunks.jsonl"])
    print("코퍼스 다운로드 완료")
else:
    print("코퍼스 이미 존재")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 42 files:   0%|          | 0/42 [00:00<?, ?it/s]

벡터 DB 다운로드 완료
코퍼스 이미 존재


## 3. 모델 로드 (Gemma 4 12B, 4bit NF4)

In [5]:
from transformers import AutoTokenizer, BitsAndBytesConfig, Gemma4ForCausalLM

MODEL_NAME = "google/gemma-4-12b-it"

# Colab T4(Turing, sm_75)는 bfloat16 텐서코어가 없으므로 float16을 사용한다.
# (bf16은 Ampere+ 전용 — T4에서 쓰면 에뮬레이션으로 느려지거나 일부 커널에서 에러)
COMPUTE_DTYPE = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

print("[1/2] 토크나이저 로드")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 텍스트 RAG 챗봇이므로 vision/audio 인코더를 제외한 텍스트 백본(Gemma4ForCausalLM)만 적재한다.
# → 15GB T4에서 로드 VRAM을 크게 줄여 OOM을 방지. (멀티모달 입력은 사용하지 않음)
print("[2/2] 모델 로드 (텍스트 백본, 4bit NF4)")
model = Gemma4ForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=COMPUTE_DTYPE,
)
model.eval()

vram_gb = torch.cuda.memory_reserved() / 1024**3
print(f"\n모델 로드 완료 — VRAM: {vram_gb:.2f} GB")

[1/2] 토크나이저 로드


config.json:   0%|          | 0.00/4.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/17.5k [00:00<?, ?B/s]

[2/2] 모델 로드 (텍스트 백본, 4bit NF4)


[transformers] You are using a model of type `gemma4_unified` to instantiate a model of type `gemma4_text`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:744: UserWarning: Not enough free disk space to download the file. The expected file size is: 23919.55 MB. The target location /content/drive/MyDrive/hf_cache/hub/models--google--gemma-4-12b-it/blobs only has 16090.87 MB free disk space.
  warnings.warn(


model.safetensors:   0%|          | 0.00/23.9G [00:00<?, ?B/s]

AttributeError: 'dict' object has no attribute 'to_dict'

## 4. RAG 검색기 (bge-m3 CPU)

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

print("임베딩 모델 로드: BAAI/bge-m3 (CPU)")
embed_model = SentenceTransformer("BAAI/bge-m3", device="cpu")

client = chromadb.PersistentClient(path=VECTOR_DB_PATH)
collection = client.get_collection("cnu_chunks")
print(f"벡터 DB: {collection.count()}개 청크")


def retrieve(query, top_k=5):
    query_emb = embed_model.encode([query]).tolist()[0]
    return collection.query(
        query_embeddings=[query_emb],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )


def build_context(query, top_k=5):
    results = retrieve(query, top_k)
    context_parts, urls = [], []
    for i in range(len(results["ids"][0])):
        meta = results["metadatas"][0][i]
        context_parts.append(f"[참고{i+1}] {meta['title']}\n{results['documents'][0][i]}")
        if meta["url"] and meta["url"] not in urls:
            urls.append(meta["url"])
    return "\n\n".join(context_parts), urls


# 검색 테스트
test = retrieve("졸업 요건이 어떻게 되나요?")
for i, doc in enumerate(test["documents"][0]):
    print(f"  [{i+1}] (dist={test['distances'][0][i]:.3f}) {doc[:80]}...")

## 5. 추론 함수

In [ ]:
KST = timezone(timedelta(hours=9))
now = datetime.now(KST)
today = now.strftime("%Y-%m-%d (%A)")
month = now.month
if 3 <= month <= 8:
    semester = f"{now.year}학년도 1학기"
else:
    year = now.year if month >= 9 else now.year - 1
    semester = f"{year}학년도 2학기"

SYSTEM_PROMPT = (
    f"너는 충남대학교 학내 정보를 안내하는 친절한 AI 챗봇이야.\n"
    f"오늘 날짜: {today} | 현재 학기: {semester}\n\n"
    "대화 스타일:\n"
    "- 친근하고 자연스러운 말투로 대답해. 딱딱하지 않게, 친구에게 설명하듯이.\n"
    "- '~해요', '~이에요' 같은 존댓말을 사용하되 부드럽게.\n"
    "- 질문에 맞는 핵심 정보를 먼저 알려주고, 필요하면 추가 설명을 덧붙여.\n\n"
    "규칙:\n"
    "1. 주어진 참고 자료에 있는 정보를 기반으로 답변해. 참고 자료에 날짜, 학점, 일정 등 구체적 수치가 있으면 반드시 포함해서 답해.\n"
    "2. 참고 자료에 없는 내용은 절대 지어내지 마. '해당 정보를 찾지 못했어요'라고 솔직히 답해.\n"
    "3. 기숙사 식단과 학생회관 식단을 혼동하지 마.\n"
    "4. 사용자가 점심만 물어보면 점심만 답해.\n"
    "5. 참고 자료의 원본 데이터를 그대로 전달해. 메뉴명, 일정명 등 고유명사에 형용사나 수식어를 절대 추가하지 마. 없는 메뉴나 일정을 지어내지 마.\n"
    "6. 반드시 문법적으로 자연스러운 한국어로만 답변해. 중국어, 영어, 일본어 등 다른 언어를 절대 사용하지 마.\n"
    "7. '이번 학기'는 현재 학기를 의미하고, '다음 학기'는 그 다음 학기를 의미해.\n"
    "8. 공지사항이나 목록을 보여줄 때는 참고 자료에 나온 순서대로(위에서부터) 답변해. 임의로 순서를 바꾸지 마."
)

_SOURCE_LABELS = {
    "computer.cnu.ac.kr": "컴퓨터융합학부",
    "plus.cnu.ac.kr": "충남대 공식",
    "job.cnu.ac.kr": "인재개발원",
    "sugang.cnu.ac.kr": "수강신청",
    "www.cnucoop.co.kr": "생활협동조합",
    "mobileadmin.cnu.ac.kr": "충남대 식단",
}


def _format_sources(urls):
    if not urls:
        return ""
    seen = []
    for url in urls:
        for domain, label in _SOURCE_LABELS.items():
            if domain in url:
                if label not in seen:
                    seen.append(label)
                break
        else:
            host = urlparse(url).hostname or url
            short = host.replace("www.", "").split(".")[0]
            if short not in seen:
                seen.append(short)
    return "\n\n출처: " + ", ".join(seen)


def _clean_thinking(text):
    """Gemma 4 thinking 블록 잔여물 제거 (안전망).

    실제 thinking 비활성화는 apply_chat_template(enable_thinking=False)로 한다.
    여기서는 혹시 남은 <think>...</think> 블록만 제거하고, 정상 답변 본문은
    절대 잘라내지 않는다.
    """
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    # 닫는 태그 없이 <think>만 남은 경우: 해당 태그 이후 전부 thinking으로 보고 제거
    text = re.sub(r"<think>.*$", "", text, flags=re.DOTALL)
    return text.strip()


def generate_answer(question, max_new_tokens=1024):
    gc.collect()
    torch.cuda.empty_cache()

    context, urls = build_context(question, top_k=5)
    if not context:
        return "관련 정보를 찾을 수 없습니다."

    user_msg = (
        "아래 참고 자료를 반드시 읽고, 참고 자료에 있는 내용만으로 답변해.\n\n"
        f"참고 자료:\n{context}\n\n질문: {question}"
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]

    # enable_thinking=False: Gemma 4 thinking(추론) 모드 비활성화.
    # (이전 코드의 thinking=False는 잘못된 인자명이라 무시되어 thinking이 켜져 있었음)
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.3,
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    answer = _clean_thinking(answer)
    answer += _format_sources(urls)

    del inputs, outputs, generated_ids
    gc.collect()
    torch.cuda.empty_cache()
    return answer


# 테스트
print(generate_answer("수강신청은 어떻게 하나요?"))

## 6. 배치 추론

In [ ]:
def run_batch(test_path, output_path):
    if not os.path.exists(test_path):
        print(f"{test_path} 없음 — 건너뜀")
        return
    with open(test_path, encoding="utf-8") as f:
        data = json.load(f)
    os.makedirs("outputs", exist_ok=True)
    results = []
    for i, item in enumerate(data, 1):
        q = item["user"]
        print(f"[{i}/{len(data)}] {q[:50]}...", end=" ", flush=True)
        start = time.time()
        answer = generate_answer(q)
        print(f"{time.time()-start:.1f}s")
        results.append({"user": q, "model": answer})
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"저장: {output_path} ({len(results)}건)")


run_batch("data/test_chat.json", "outputs/chat_output.json")
run_batch("data/test_realtime.json", "outputs/realtime_output.json")

## 7. Gradio UI

In [ ]:
import gradio as gr

with gr.Blocks(title="충남대 Q&A") as demo:
    gr.Markdown("# 충남대학교 학내 정보 Q&A\n학사, 식단, 셔틀 등을 질문해보세요.")
    q_input = gr.Textbox(label="질문", placeholder="예: 졸업 요건이 뭐야?", lines=2)
    btn = gr.Button("질문하기", variant="primary")
    a_output = gr.Textbox(label="답변", lines=10, interactive=False)
    gr.Examples(
        examples=["컴퓨터융합학부 졸업 요건", "수강신청 언제?", "제1학생회관 점심 메뉴",
                  "셔틀버스 시간표", "장학금 신청 방법"],
        inputs=q_input,
    )
    btn.click(fn=generate_answer, inputs=q_input, outputs=a_output)
    q_input.submit(fn=generate_answer, inputs=q_input, outputs=a_output)

demo.launch(share=True)

## 8. 결과 확인

In [ ]:
for path in ["outputs/chat_output.json", "outputs/realtime_output.json"]:
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        print(f"\n{'='*60}\n{path} — {len(data)}건\n{'='*60}")
        for item in data[:3]:
            print(f"Q: {item['user'][:80]}")
            print(f"A: {item['model'][:200]}...\n")
    else:
        print(f"{path} — 없음")